In [ ]:
# connect to data in google drive

from google.colab import drive
drive.mount('/content/drive')

In [7]:
# FINE TUNE MODEL

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import os

# ==========================================
# 1. SETUP PATHS & CONFIGURATIONS
# ==========================================
# UPDATED: Changed path to use your fresh auto-split directory
DATA_DIR = '/content/drive/MyDrive/hackathon-palm-oil/split_dataset-rnn'
BATCH_SIZE = 32
EPOCHS = 10
CLASSES = ['underripe', 'ripe', 'overripe', 'rotten']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# ==========================================
# 2. DEFINE DATA TRANSFORMS & LOADERS
# ==========================================
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# UPDATED: Separate loaders for both the Training and Validation folders
train_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'train'), transform=data_transforms)
val_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'val'), transform=data_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Loaded {len(train_dataset)} training images across classes: {train_dataset.classes}")
print(f"Loaded {len(val_dataset)} validation images across classes: {val_dataset.classes}")

# ==========================================
# 3. INITIALIZE MODEL ARCHITECTURE
# ==========================================
def load_vision_grader_model(num_classes=3):
    weights = models.EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)

    for param in model.parameters():
        param.requires_grad = True

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model.to(device)

model = load_vision_grader_model(num_classes=len(CLASSES))

# ==========================================
# 4. DEFINE LOSS & OPTIMIZER
# ==========================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==========================================
# 5. THE TRAINING & VALIDATION LOOP ENGINE
# ==========================================
print("\nStarting model training...")
for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = (correct / total) * 100

    # --- UPDATED: VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_epoch_loss = val_loss / len(val_dataset)
    val_epoch_acc = (val_correct / val_total) * 100

    # Print combined metrics for clear tracking
    print(f"Epoch [{epoch+1}/{EPOCHS}] -> "
          f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:.2f}%")

print("\nTraining complete! Your CNN model is fully trained.")


Using device: cuda
GPU Name: Tesla T4
Loaded 2276 training images across classes: ['overripe', 'ripe', 'rotten', 'underripe']
Loaded 205 validation images across classes: ['overripe', 'ripe', 'underripe']
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 152MB/s]



Starting model training...
Epoch [1/10] -> Train Loss: 0.4317 | Train Acc: 83.88% | Val Loss: 3.9075 | Val Acc: 45.85%
Epoch [2/10] -> Train Loss: 0.2239 | Train Acc: 92.49% | Val Loss: 4.1032 | Val Acc: 57.56%
Epoch [3/10] -> Train Loss: 0.1558 | Train Acc: 94.68% | Val Loss: 3.8822 | Val Acc: 53.66%
Epoch [4/10] -> Train Loss: 0.1338 | Train Acc: 94.90% | Val Loss: 3.6405 | Val Acc: 56.59%
Epoch [5/10] -> Train Loss: 0.1417 | Train Acc: 95.91% | Val Loss: 3.7901 | Val Acc: 53.17%
Epoch [6/10] -> Train Loss: 0.1177 | Train Acc: 96.09% | Val Loss: 6.1093 | Val Acc: 58.05%
Epoch [7/10] -> Train Loss: 0.0591 | Train Acc: 98.42% | Val Loss: 5.3343 | Val Acc: 55.61%
Epoch [8/10] -> Train Loss: 0.0702 | Train Acc: 97.58% | Val Loss: 5.0727 | Val Acc: 56.10%
Epoch [9/10] -> Train Loss: 0.0745 | Train Acc: 97.80% | Val Loss: 6.0862 | Val Acc: 54.15%
Epoch [10/10] -> Train Loss: 0.0650 | Train Acc: 98.20% | Val Loss: 4.9980 | Val Acc: 52.68%

Training complete! Your CNN model is fully trained

In [ ]:
# INFERENCE HERE --

import torch
from torchvision import transforms
from PIL import Image

# 1. Configuration (Must match your training setup exactly)
CLASSES = ['underripe', 'ripe', 'overripe']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Define the exact inference transform pipeline
inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def predict_ripeness(image_path, model):
    """Loads an image, runs inference, and returns the predicted class."""
    # Set model to evaluation mode
    model.eval()

    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    input_tensor = inference_transforms(image).unsqueeze(0).to(device)

    # Disable gradient calculation for speed
    with torch.no_grad():
        outputs = model(input_tensor)
        # FIXED: Removed the [0] index so it correctly applies softmax across the batch dimension
        probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]
        confidence, predicted_idx = torch.max(probabilities, 0)

    predicted_class = CLASSES[predicted_idx.item()]
    print(f"🔮 Prediction: {predicted_class.upper()} ({confidence.item()*100:.2f}% Confidence)")
    return predicted_class

# 3. Test a single image
# CHANGE THIS to your uploaded test image path inside Colab
test_image_path = '/content/split_dataset/test/ripe/your_test_photo.jpg'

# Run prediction
predict_ripeness(test_image_path, model)


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [ ]:
import os

# Update to your actual dataset path containing the train folder
DATA_DIR = '/content/drive/MyDrive/hackathon-palm-oil/split_dataset-rnn/train'

# Add 'rotten' to your classes list
classes = ['underripe', 'ripe', 'overripe', 'rotten']

print("📊 COUNTS PER FOLDER:")
print("-" * 25)

for cls in classes:
    folder_path = os.path.join(DATA_DIR, cls)

    if os.path.exists(folder_path):
        # Ignore hidden system files and count only actual image files
        files = [f for f in os.listdir(folder_path) if not f.startswith('.')]
        print(f"📁 {cls.upper()}: {len(files)} images")
    else:
        print(f"❌ Folder '{cls}' not found at {folder_path}")

📊 COUNTS PER FOLDER:
-------------------------
📁 UNDERRIPE: 635 images
📁 RIPE: 552 images
📁 OVERRIPE: 457 images
📁 ROTTEN: 96 images


In [ ]:
# AUTO SPLIT DATA INTO FOLDERS (VALIDATION AND TEST FOLDERS)

import os
import shutil
import random

# Configuration
source_dir = '/content/train'  # Your current folder with underripe, ripe, overripe
output_dir = '/content/split_dataset'
classes = ['underripe', 'ripe', 'overripe']

# Create new destination folders
for split in ['train', 'val', 'test']:
    for cls in classes:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

# Execute the 80-10-10 calculation and copy files
random.seed(42) # Keeps the split consistent every time you run it
for cls in classes:
    cls_path = os.path.join(source_dir, cls)
    images = [f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f)) and not f.startswith('.')]

    random.shuffle(images)

    total = len(images)
    train_end = int(total * 0.80)
    val_end = train_end + int(total * 0.10)

    # Separate the lists
    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    # Physically move copies to split folders
    for img in train_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(output_dir, 'train', cls, img))
    for img in val_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(output_dir, 'val', cls, img))
    for img in test_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(output_dir, 'test', cls, img))

print("📊 Dataset successfully split into: 80% Train, 10% Validation, and 10% Test!")


📊 Dataset successfully split into: 80% Train, 10% Validation, and 10% Test!


In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import os

# 1. Configuration (Must match your training directory)
DATA_DIR = '/content/split_dataset'
BATCH_SIZE = 32
CLASSES = ['underripe', 'ripe', 'overripe']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating on device: {device}")

# 2. Set up the Validation Transform Pipeline and Loader
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'val'), transform=val_transforms)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Testing model against {len(val_dataset)} total images in the validation set...\n")

# 3. Validation Evaluation Loop
criterion = nn.CrossEntropyLoss()
model.eval()  # Set model to evaluation mode

val_loss = 0.0
correct = 0
total = 0

# Disable tracking gradients to maximize inference speed and save RAM memory
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Calculate loss and tracking statistics
        val_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

# Compute final overall scores
final_loss = val_loss / len(val_dataset)
final_accuracy = (correct / total) * 100

print("📊 FINAL VALIDATION RESULTS:")
print("-" * 30)
print(f"📉 Total Validation Loss: {final_loss:.4f}")
print(f"🎯 Total Validation Accuracy: {final_accuracy:.2f}% ({correct}/{total} images correct)")


Evaluating on device: cuda
Testing model against 205 total images in the validation set...

📊 FINAL VALIDATION RESULTS:
------------------------------
📉 Total Validation Loss: 0.3863
🎯 Total Validation Accuracy: 92.68% (190/205 images correct)


In [ ]:
print(BATCH_SIZE)

32


In [ ]:
# RUN ON TEST CODE

import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import os

# 1. Configuration (Points directly to your test split folder)
DATA_DIR = '/content/split_dataset'
BATCH_SIZE = 32
CLASSES = ['underripe', 'ripe', 'overripe']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating on device: {device}")

# 2. Set up the Test Transform Pipeline and Loader
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'test'), transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Testing model against {len(test_dataset)} total images in the hidden test set...\n")

# 3. Test Evaluation Loop
criterion = nn.CrossEntropyLoss()
model.eval()  # Keep model in evaluation mode

test_loss = 0.0
correct = 0
total = 0

# Disable tracking gradients for maximum speed
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Calculate loss and tracking statistics
        test_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

# Compute final overall test scores
final_loss = test_loss / len(test_dataset)
final_accuracy = (correct / total) * 100

print("📊 FINAL HIDDEN TEST RESULTS:")
print("-" * 30)
print(f"📉 Total Test Loss: {final_loss:.4f}")
print(f"🎯 Total Test Accuracy: {final_accuracy:.2f}% ({correct}/{total} images correct)")


Evaluating on device: cuda
Testing model against 208 total images in the hidden test set...

📊 FINAL HIDDEN TEST RESULTS:
------------------------------
📉 Total Test Loss: 0.3197
🎯 Total Test Accuracy: 93.27% (194/208 images correct)


In [8]:
# DONWLOAD MODEL DIRECTLY ON LOCAL COMPUTER

import torch

# 1. Save the model's weights into a file
torch.save(model.state_dict(), 'efficientnet_palm_grader.pth')
print("Model weights successfully saved locally in Colab files!")

# 2. Download it automatically to your laptop
from google.colab import files
files.download('efficientnet_palm_grader.pth')


Model weights successfully saved locally in Colab files!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# COPY THE DATASET INTO GDRIVE FOR REFERNCE PURPOSES

# 1. Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Copy the folder into your Google Drive root folder
!cp -r split_dataset /content/drive/MyDrive/hackathon-palm-oil


Mounted at /content/drive


In [ ]:
# train + save all 3 model of xgboost

import pandas as pd
import xgboost as xgb

df = pd.read_csv("synthetic_palm_oil_quality_dataset.csv")

FEATURES = ['ripeness_score', 'harvest_delay_hrs', 'storage_temp_c', 'humidity_pct']
X = df[FEATURES]

# ---- Model 1: FFA ----
y_ffa = df['FFA_actual']
ffa_model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05)
ffa_model.fit(X, y_ffa)
ffa_model.save_model("ffa_model.json")

# ---- Model 2: Moisture ----
y_moisture = df['moisture_actual']
moisture_model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05)
moisture_model.fit(X, y_moisture)
moisture_model.save_model("moisture_model.json")

# ---- Model 3: Purity ----
y_purity = df['purity_actual']
purity_model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05)
purity_model.fit(X, y_purity)
purity_model.save_model("purity_model.json")

print("All 3 models trained and saved.")

All 3 models trained and saved.


In [ ]:
# test all 3 model of xgboost

from sklearn.metrics import mean_absolute_error, r2_score

test_df = pd.read_csv("test_dataset_with_answers.csv")
X_test = test_df[FEATURES]

for target, model in [("FFA_actual", ffa_model), ("moisture_actual", moisture_model), ("purity_actual", purity_model)]:
    y_true = test_df[target]
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{target}: MAE={mae:.4f}, R2={r2:.4f}")

FFA_actual: MAE=0.1386, R2=0.9275
moisture_actual: MAE=0.3277, R2=0.9403
purity_actual: MAE=0.1118, R2=0.9289
